# Project: **AUTONOMOUS CAR SYSTEM AND LANEDETECTION** 

**In this project, I used Python and OpenCV to detect lane lines on the road.
I developed a processing pipeline that works on a series of individual images, and applied the result to a video stream.**

**Example of the intended output**

---

<figure>
 <img src="output_example.jpg" width="380" alt="Combined Image" />
 <figcaption>
 <p></p> 
 </figcaption>
</figure>

## Pipeline architecture:
-  **Load test images.**
-  **Apply Color Selection**
-  **Apply Canny edge detection.**
    -  Apply gray scaling to the images.
    -  Apply Gaussian smoothing.
    -  Perform Canny edge detection.
-  **Determine the region of interest.**
-  **Apply Hough transform.**
-  **Average and extrapolating the lane lines.**
-  **Apply on video streams.**

I'll explain each step in details below.

#### Environement:
-  Windows 7
-  Anaconda 4.3.29
-  Python 3.6.2
-  OpenCV 3.1.0

## 1. Loading test images

 We will write a function called `list_images()` that will show all the test images we're working on.

In [ ]:
#Importing some useful packages
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import cv2
import os
import glob
from moviepy.editor import VideoFileClip



In [ ]:
def list_images(images, cols = 2, rows = 5, cmap=None):
    """
    Display a list of images in a single figure with matplotlib.
        Parameters:
            images: List of np.arrays compatible with plt.imshow.
            cols (Default = 2): Number of columns in the figure.
            rows (Default = 5): Number of rows in the figure.
            cmap (Default = None): Used to display gray images.
    """
    plt.figure(figsize=(10, 11))
    for i, image in enumerate(images):
        plt.subplot(rows, cols, i+1)
        #Use gray scale color map if there is only one channel
        cmap = 'gray' if len(image.shape) == 2 else cmap
        plt.imshow(image, cmap = cmap)
        plt.xticks([])
        plt.yticks([])
    plt.tight_layout(pad=0, h_pad=0, w_pad=0)
    plt.show()

In [ ]:
import glob
import matplotlib.pyplot as plt

def list_images(images, cols=2, rows=5, cmap=None):
    """
    Display a list of images in a single figure with matplotlib.
        Parameters:
            images: List of np.arrays compatible with plt.imshow.
            cols (Default = 2): Number of columns in the figure.
            rows (Default = 5): Number of rows in the figure.
            cmap (Default = None): Used to display gray images.
    """
    plt.figure(figsize=(10, 11))
    for i, image in enumerate(images):
        plt.subplot(rows, cols, i+1)
        #Use gray scale color map if there is only one channel
        cmap = 'gray' if len(image.shape) == 2 else cmap
        plt.imshow(image, cmap=cmap)
        plt.xticks([])
        plt.yticks([])
    plt.tight_layout(pad=0, h_pad=0, w_pad=0)
    plt.show()

test_images = [plt.imread(img) for img in glob.glob('test_images/02. solidWhiteRight.jpg')]

print(len(test_images))  # should be > 5

list_images(test_images)

## 2. Color Selection

Lane lines in the test images are in white and yellow. We need to choose the most suitable color space, that clearly highlights the lane lines.

### Original RGB color selection

I will apply color selection to the `test_images` in the original RGB format. We will try to retain as much of the lane lines as possible, while blacking out most of the other stuff.

In [ ]:
def RGB_color_selection(image):
    """
    Apply color selection to RGB images to blackout everything except for white and yellow lane lines.
        Parameters:
            image: An np.array compatible with plt.imshow.
    """
    # Ensure image is RGB (3 channels)
    if len(image.shape) == 2:
        # Convert grayscale to RGB
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    elif image.shape[2] == 4:
        # Convert RGBA to RGB
        image = cv2.cvtColor(image, cv2.COLOR_RGBA2RGB)
    
    #White color mask
    lower_threshold = np.uint8([200, 200, 200])
    upper_threshold = np.uint8([255, 255, 255])
    white_mask = cv2.inRange(image, lower_threshold, upper_threshold)
    
    #Yellow color mask
    lower_threshold = np.uint8([175, 175, 0])
    upper_threshold = np.uint8([255, 255, 255])
    yellow_mask = cv2.inRange(image, lower_threshold, upper_threshold)
    
    #Combine white and yellow masks
    mask = cv2.bitwise_or(white_mask, yellow_mask)
    masked_image = cv2.bitwise_and(image, image, mask=mask)
    
    return masked_image

Applying color selection to `test_images` in the RGB color space.

In [ ]:
import numpy as np
import cv2

def RGB_color_selection(image):
	"""
	Apply color selection to RGB images to blackout everything except for white and yellow lane lines.
		Parameters:
			image: An np.array compatible with plt.imshow.
	"""
	# Ensure image is RGB (3 channels)
	if len(image.shape) == 2:
		# Convert grayscale to RGB
		image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
	elif image.shape[2] == 4:
		# Convert RGBA to RGB
		image = cv2.cvtColor(image, cv2.COLOR_RGBA2RGB)
	
	#White color mask
	lower_threshold = np.uint8([200, 200, 200])
	upper_threshold = np.uint8([255, 255, 255])
	white_mask = cv2.inRange(image, lower_threshold, upper_threshold)
	
	#Yellow color mask
	lower_threshold = np.uint8([175, 175, 0])
	upper_threshold = np.uint8([255, 255, 255])
	yellow_mask = cv2.inRange(image, lower_threshold, upper_threshold)
	
	#Combine white and yellow masks
	mask = cv2.bitwise_or(white_mask, yellow_mask)
	masked_image = cv2.bitwise_and(image, image, mask=mask)
	
	return masked_image

list_images(list(map(RGB_color_selection, test_images)))

### a) HSV color space

**Wikipedia**: HSV is an alternative representation of the RGB color model. The HSV representation models the way colors mix together, with the saturation dimension resembling various shades of brightly colored paint, and the value dimension resembling the mixture of those paints with varying amounts of black or white.

In [ ]:
def convert_hsv(image):
    """
    Convert RGB images to HSV.
        Parameters:
            image: An np.array compatible with plt.imshow.
    """
    return cv2.cvtColor(image, cv2.COLOR_RGB2HSV)

list_images(list(map(convert_hsv, test_images)))

In [ ]:
def HSV_color_selection(image):
    """
    Apply color selection to the HSV images to blackout everything except for white and yellow lane lines.
        Parameters:
            image: An np.array compatible with plt.imshow.
    """
    #Convert the input image to HSV
    converted_image = convert_hsv(image)
    
    #White color mask
    lower_threshold = np.uint8([0, 0, 210])
    upper_threshold = np.uint8([255, 30, 255])
    white_mask = cv2.inRange(converted_image, lower_threshold, upper_threshold)
    
    #Yellow color mask
    lower_threshold = np.uint8([18, 80, 80])
    upper_threshold = np.uint8([30, 255, 255])
    yellow_mask = cv2.inRange(converted_image, lower_threshold, upper_threshold)
    
    #Combine white and yellow masks
    mask = cv2.bitwise_or(white_mask, yellow_mask)
    masked_image = cv2.bitwise_and(image, image, mask = mask)
    
    return masked_image

Applying color selection to `test_images` in the HSV color space.

In [ ]:
list_images(list(map(HSV_color_selection, test_images)))

### b) HSL color space

**Wikipedia**: HSL is an alternative representation of the RGB color model. The HSL model attempts to resemble more perceptual color models such as NCS or Munsell, placing fully saturated colors around a circle at a lightness value of 1/2, where a lightness value of 0 or 1 is fully black or white, respectively.

In [ ]:
def convert_hsl(image):
    """
    Convert RGB images to HSL.
        Parameters:
            image: An np.array compatible with plt.imshow.
    """
    return cv2.cvtColor(image, cv2.COLOR_RGB2HLS)

list_images(list(map(convert_hsl, test_images)))

In [ ]:
def HSL_color_selection(image):
    """
    Apply color selection to the HSL images to blackout everything except for white and yellow lane lines.
        Parameters:
            image: An np.array compatible with plt.imshow.
    """
    #Convert the input image to HSL
    converted_image = convert_hsl(image)
    
    #White color mask
    lower_threshold = np.uint8([0, 200, 0])
    upper_threshold = np.uint8([255, 255, 255])
    white_mask = cv2.inRange(converted_image, lower_threshold, upper_threshold)
    
    #Yellow color mask
    lower_threshold = np.uint8([10, 0, 100])
    upper_threshold = np.uint8([40, 255, 255])
    yellow_mask = cv2.inRange(converted_image, lower_threshold, upper_threshold)
    
    #Combine white and yellow masks
    mask = cv2.bitwise_or(white_mask, yellow_mask)
    masked_image = cv2.bitwise_and(image, image, mask = mask)
    
    return masked_image

Applying color selection to `test_images` in the HSL color space.

In [ ]:
list_images(list(map(HSL_color_selection, test_images)))

Using HSL produces the clearest lane lines of all color spaces. We will use them for the next steps.

In [ ]:
color_selected_images = list(map(HSL_color_selection, test_images))

## 3. Canny Edge Detection

**Wikipedia**: The Canny edge detector is an edge detection operator that uses a multi-stage algorithm to detect a wide range of edges in images.

### a) Gray scaling the images

The Canny edge detection algorithm measures the intensity gradients of each pixel. So, we need to convert the images into gray scale in order to detect edges.

In [ ]:
def gray_scale(image):
    """
    Convert images to gray scale.
        Parameters:
            image: An np.array compatible with plt.imshow.
    """
    return cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

In [ ]:
def gray_scale(image):
	"""
	Convert images to gray scale.
		Parameters:
			image: An np.array compatible with plt.imshow.
	"""
	return cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

gray_images = list(map(gray_scale, color_selected_images))
list_images(gray_images)

### b) Applying Gaussian smoothing

**Wikipedia**: Since all edge detection results are easily affected by image noise, it is essential to filter out the noise to prevent false detection caused by noise. To smooth the image, a Gaussian filter is applied to convolve with the image. This step will slightly smooth the image to reduce the effects of obvious noise on the edge detector.

In [ ]:
def gaussian_smoothing(image, kernel_size = 13):
    """
    Apply Gaussian filter to the input image.
        Parameters:
            image: An np.array compatible with plt.imshow.
            kernel_size (Default = 13): The size of the Gaussian kernel will affect the performance of the detector.
            It must be an odd number (3, 5, 7, ...).
    """
    return cv2.GaussianBlur(image, (kernel_size, kernel_size), 0)

In [ ]:
def gaussian_smoothing(image, kernel_size=13):
	"""
	Apply Gaussian filter to the input image.
		Parameters:
			image: An np.array compatible with plt.imshow.
			kernel_size (Default = 13): The size of the Gaussian kernel will affect the performance of the detector.
			It must be an odd number (3, 5, 7, ...).
	"""
	return cv2.GaussianBlur(image, (kernel_size, kernel_size), 0)

blur_images = list(map(gaussian_smoothing, gray_images))
list_images(blur_images)

### c) Applying Canny Edge Detection

**Wikipedia**:
The Process of Canny edge detection algorithm can be broken down to 5 different steps:
1. Find the intensity gradients of the image
2. Apply non-maximum suppression to get rid of spurious response to edge detection.
3. Apply *double threshold* to determine potential edges.
4. Track edge by hysteresis: Finalize the detection of edges by suppressing all the other edges that are weak and not connected to strong edges.

**If an edge pixel’s gradient value is higher than the high threshold value, it is marked as a strong edge pixel. If an edge pixel’s gradient value is smaller than the high threshold value and larger than the low threshold value, it is marked as a weak edge pixel. If an edge pixel's value is smaller than the low threshold value, it will be suppressed.
The two threshold values are empirically determined and their definition will depend on the content of a given input image.*

In [ ]:
def canny_detector(image, low_threshold = 50, high_threshold = 150):
    """
    Apply Canny Edge Detection algorithm to the input image.
        Parameters:
            image: An np.array compatible with plt.imshow.
            low_threshold (Default = 50).
            high_threshold (Default = 150).
    """
    return cv2.Canny(image, low_threshold, high_threshold)

In [ ]:
def canny_detector(image, low_threshold=50, high_threshold=150):
	"""
	Apply Canny Edge Detection algorithm to the input image.
		Parameters:
			image: An np.array compatible with plt.imshow.
			low_threshold (Default = 50).
			high_threshold (Default = 150).
	"""
	return cv2.Canny(image, low_threshold, high_threshold)

edge_detected_images = list(map(canny_detector, blur_images))
list_images(edge_detected_images)

## 4. Region of interest

We're interested in the area facing the camera, where the lane lines are found. So, we'll apply region masking to cut out everything else.

In [ ]:
def region_selection(image):
    """
    Determine and cut the region of interest in the input image.
        Parameters:
            image: An np.array compatible with plt.imshow.
    """
    mask = np.zeros_like(image)   
    #Defining a 3 channel or 1 channel color to fill the mask with depending on the input image
    if len(image.shape) > 2:
        channel_count = image.shape[2]
        ignore_mask_color = (255,) * channel_count
    else:
        ignore_mask_color = 255
    #We could have used fixed numbers as the vertices of the polygon,
    #but they will not be applicable to images with different dimesnions.
    rows, cols = image.shape[:2]
    bottom_left  = [cols * 0.1, rows * 0.95]
    top_left     = [cols * 0.4, rows * 0.6]
    bottom_right = [cols * 0.9, rows * 0.95]
    top_right    = [cols * 0.6, rows * 0.6]
    vertices = np.array([[bottom_left, top_left, top_right, bottom_right]], dtype=np.int32)
    cv2.fillPoly(mask, vertices, ignore_mask_color)
    masked_image = cv2.bitwise_and(image, mask)
    return masked_image

In [ ]:
def region_selection(image):
	"""
	Determine and cut the region of interest in the input image.
		Parameters:
			image: An np.array compatible with plt.imshow.
	"""
	mask = np.zeros_like(image)   
	#Defining a 3 channel or 1 channel color to fill the mask with depending on the input image
	if len(image.shape) > 2:
		channel_count = image.shape[2]
		ignore_mask_color = (255,) * channel_count
	else:
		ignore_mask_color = 255
	#We could have used fixed numbers as the vertices of the polygon,
	#but they will not be applicable to images with different dimesnions.
	rows, cols = image.shape[:2]
	bottom_left  = [cols * 0.1, rows * 0.95]
	top_left     = [cols * 0.4, rows * 0.6]
	bottom_right = [cols * 0.9, rows * 0.95]
	top_right    = [cols * 0.6, rows * 0.6]
	vertices = np.array([[bottom_left, top_left, top_right, bottom_right]], dtype=np.int32)
	cv2.fillPoly(mask, vertices, ignore_mask_color)
	masked_image = cv2.bitwise_and(image, mask)
	return masked_image

masked_image = list(map(region_selection, edge_detected_images))
list_images(masked_image)

## 5. Hough Transform

The Hough transform is a technique which can be used to isolate features of a particular shape within an image. I'll use it to detected the lane lines in `selected_region_images`.

In [ ]:
def hough_transform(image):
    """
    Determine and cut the region of interest in the input image.
        Parameters:
            image: The output of a Canny transform.
    """
    rho = 1              #Distance resolution of the accumulator in pixels.
    theta = np.pi/180    #Angle resolution of the accumulator in radians.
    threshold = 20       #Only lines that are greater than threshold will be returned.
    minLineLength = 20   #Line segments shorter than that are rejected.
    maxLineGap = 300     #Maximum allowed gap between points on the same line to link them
    return cv2.HoughLinesP(image, rho = rho, theta = theta, threshold = threshold,
                           minLineLength = minLineLength, maxLineGap = maxLineGap)

`hough_lines` contains the list of lines detected in the selected region. Now, we will draw these detected lines onto the original `test_images`.

In [ ]:
hough_lines = list(map(hough_transform, masked_image))

In [ ]:
def draw_lines(image, lines, color = [255, 0, 0], thickness = 2):
    """
    Draw lines onto the input image.
        Parameters:
            image: An np.array compatible with plt.imshow.
            lines: The lines we want to draw.
            color (Default = red): Line color.
            thickness (Default = 2): Line thickness.
    """
    image = np.copy(image)
    for line in lines:
        for x1,y1,x2,y2 in line:
            cv2.line(image, (x1, y1), (x2, y2), color, thickness)
    return image

In [ ]:
def draw_lines(image, lines, color = [255, 0, 0], thickness = 2):
    """
    Draw lines onto the input image.
        Parameters:
            image: An np.array compatible with plt.imshow.
            lines: The lines we want to draw.
            color (Default = red): Line color.
            thickness (Default = 2): Line thickness.
    """
    image = np.copy(image)
    for line in lines:
        for x1,y1,x2,y2 in line:
            cv2.line(image, (x1, y1), (x2, y2), color, thickness)
    return image

def hough_transform(image):
    """
    Determine and cut the region of interest in the input image.
        Parameters:
            image: The output of a Canny transform.
    """
    rho = 1              #Distance resolution of the accumulator in pixels.
    theta = np.pi/180    #Angle resolution of the accumulator in radians.
    threshold = 20       #Only lines that are greater than threshold will be returned.
    minLineLength = 20   #Line segments shorter than that are rejected.
    maxLineGap = 300     #Maximum allowed gap between points on the same line to link them
    return cv2.HoughLinesP(image, rho = rho, theta = theta, threshold = threshold,
                           minLineLength = minLineLength, maxLineGap = maxLineGap)

hough_lines = list(map(hough_transform, masked_image))

line_images = []
for image, lines in zip(test_images, hough_lines):
    line_images.append(draw_lines(image, lines))
    
list_images(line_images)

## 6. Averaging and extrapolating the lane lines

We have multiple lines detected for each lane line. We need to average all these lines and draw a single line for each lane line.
We also need to extrapolate the lane lines to cover the full lane line length.

In [ ]:
def average_slope_intercept(lines):
    """
    Find the slope and intercept of the left and right lanes of each image.
        Parameters:
            lines: The output lines from Hough Transform.
    """
    left_lines    = [] #(slope, intercept)
    left_weights  = [] #(length,)
    right_lines   = [] #(slope, intercept)
    right_weights = [] #(length,)
    
    for line in lines:
        for x1, y1, x2, y2 in line:
            if x1 == x2:
                continue
            slope = (y2 - y1) / (x2 - x1)
            intercept = y1 - (slope * x1)
            length = np.sqrt(((y2 - y1) ** 2) + ((x2 - x1) ** 2))
            if slope < 0:
                left_lines.append((slope, intercept))
                left_weights.append((length))
            else:
                right_lines.append((slope, intercept))
                right_weights.append((length))
    left_lane  = np.dot(left_weights,  left_lines) / np.sum(left_weights)  if len(left_weights) > 0 else None
    right_lane = np.dot(right_weights, right_lines) / np.sum(right_weights) if len(right_weights) > 0 else None
    return left_lane, right_lane

In [ ]:
def pixel_points(y1, y2, line):
    """
    Converts the slope and intercept of each line into pixel points.
        Parameters:
            y1: y-value of the line's starting point.
            y2: y-value of the line's end point.
            line: The slope and intercept of the line.
    """
    if line is None:
        return None
    slope, intercept = line
    x1 = int((y1 - intercept)/slope)
    x2 = int((y2 - intercept)/slope)
    y1 = int(y1)
    y2 = int(y2)
    return ((x1, y1), (x2, y2))

In [ ]:
def average_slope_intercept(lines):
    """
    Find the slope and intercept of the left and right lanes of each image.
        Parameters:
            lines: The output lines from Hough Transform.
    """
    left_lines    = [] #(slope, intercept)
    left_weights  = [] #(length,)
    right_lines   = [] #(slope, intercept)
    right_weights = [] #(length,)
    
    for line in lines:
        for x1, y1, x2, y2 in line:
            if x1 == x2:
                continue
            slope = (y2 - y1) / (x2 - x1)
            intercept = y1 - (slope * x1)
            length = np.sqrt(((y2 - y1) ** 2) + ((x2 - x1) ** 2))
            if slope < 0:
                left_lines.append((slope, intercept))
                left_weights.append((length))
            else:
                right_lines.append((slope, intercept))
                right_weights.append((length))
    left_lane  = np.dot(left_weights,  left_lines) / np.sum(left_weights)  if len(left_weights) > 0 else None
    right_lane = np.dot(right_weights, right_lines) / np.sum(right_weights) if len(right_weights) > 0 else None
    return left_lane, right_lane

def pixel_points(y1, y2, line):
    """
    Converts the slope and intercept of each line into pixel points.
        Parameters:
            y1: y-value of the line's starting point.
            y2: y-value of the line's end point.
            line: The slope and intercept of the line.
    """
    if line is None:
        return None
    slope, intercept = line
    x1 = int((y1 - intercept)/slope)
    x2 = int((y2 - intercept)/slope)
    y1 = int(y1)
    y2 = int(y2)
    return ((x1, y1), (x2, y2))

def lane_lines(image, lines):
    """
    Create full lenght lines from pixel points.
        Parameters:
            image: The input test image.
            lines: The output lines from Hough Transform.
    """
    left_lane, right_lane = average_slope_intercept(lines)
    y1 = image.shape[0]
    y2 = y1 * 0.6
    left_line  = pixel_points(y1, y2, left_lane)
    right_line = pixel_points(y1, y2, right_lane)
    return left_line, right_line

    
def draw_lane_lines(image, lines, color=[255, 0, 0], thickness=12):
    """
    Draw lines onto the input image.
        Parameters:
            image: The input test image.
            lines: The output lines from Hough Transform.
            color (Default = red): Line color.
            thickness (Default = 12): Line thickness. 
    """
    line_image = np.zeros_like(image)
    for line in lines:
        if line is not None:
            cv2.line(line_image, *line,  color, thickness)
    return cv2.addWeighted(image, 1.0, line_image, 1.0, 0.0)
             
    
lane_images = []
for image, lines in zip(test_images, hough_lines):
    lane_images.append(draw_lane_lines(image, lane_lines(image, lines)))

    
list_images(lane_images)

## 7. Apply on video streams

Now, we'll use the above functions to detect lane lines from a video stream.

In [ ]:
#Import everything needed to edit/save/watch video clips
from moviepy import *
from IPython.display import HTML
from IPython.display import Image

In [ ]:
def frame_processor(image):
    """
    Process the input frame to detect lane lines.
        Parameters:
            image: Single video frame.
    """
    color_select = HSL_color_selection(image)
    gray         = gray_scale(color_select)
    smooth       = gaussian_smoothing(gray)
    edges        = canny_detector(smooth)
    region       = region_selection(edges)
    hough        = hough_transform(region)
    result       = draw_lane_lines(image, lane_lines(image, hough))
    return result 

In [ ]:
def process_video(test_video, output_video):
    """
    Read input video stream and produce a video file with detected lane lines.
        Parameters:
            test_video: Input video.
            output_video: A video file with detected lane lines.
    """
    input_video = VideoFileClip(os.path.join('test_videos', test_video), audio=False)
    processed = input_video.fl_image(frame_processor)
    processed.write_videofile(os.path.join('output_videos', output_video), audio=False)

In [ ]:
from moviepy.editor import VideoFileClip
import os

%time process_video('solidWhiteRight.mp4', 'solidWhiteRight_output.mp4')
HTML("""
<video width="960" height="540" controls>
  <source src="{0}">
</video>
""".format("output_videos/solidWhiteRight_output.mp4"))

In [ ]:
from moviepy.editor import VideoFileClip
import os

%time process_video('solidYellowLeft.mp4', 'solidYellowLeft_output.mp4')
HTML("""
<video width="960" height="540" controls>
  <source src="{0}">
</video>
""".format("output_videos/solidYellowLeft_output.mp4"))

In [ ]:
%time process_video('challenge.mp4', 'challenge_output.mp4')
HTML("""
<video width="960" height="540" controls>
  <source src="{0}">
</video>
""".format("output_videos\challenge_output.mp4"))


## 8. Lane Confidance Visualization

In [ ]:
import joblib

# 1. Load the pre-trained Random Forest model from your file
# Make sure 'confidence_model.pkl' is in the same folder as your notebook
confidence_model = joblib.load('confidence_model.pkl')

print("✅ The Random Forest 'confidence_model' is now loaded and ready!")

In [ ]:
import cv2

image_to_use = cv2.imread("test_images/curved road.jpg")
image_to_use = cv2.cvtColor(image_to_use, cv2.COLOR_BGR2RGB)

In [ ]:
import numpy as np

X = np.array([
    [-0.7, 120, 300],
    [-0.8, 150, 250],
    [0.7, 130, 900],
    [0.9, 160, 950],
    
    # more noise
    [-0.2, 30, 400],
    [0.3, 40, 600],
    [0.0, 20, 500],
    [1.5, 10, 200]
])

y = np.array([
    0.9, 0.95, 0.92, 0.96,
    0.2, 0.3, 0.1, 0.25
])

In [ ]:
from sklearn.ensemble import RandomForestRegressor
import pickle

reg_model = RandomForestRegressor()
reg_model.fit(X, y)

with open("confidence_model.pkl", "wb") as f:
    pickle.dump(reg_model, f)

print("Confidence model ready")

In [ ]:
with open("confidence_model.pkl", "rb") as f:
    reg_model = pickle.load(f)

In [ ]:
def get_line_confidence(lines):
    
    features, raw_lines = extract_features_from_lines(lines)
    
    results = []
    
    for i in range(len(features)):
        score = reg_model.predict([features[i]])[0]
        
        x1, y1, x2, y2 = raw_lines[i]
        
        results.append({
            "line": [x1, y1, x2, y2],
            "confidence": score
        })
    
    return results

In [ ]:
def classify_confidence(score):
    if score > 0.85:
        return "High"
    elif score > 0.6:
        return "Medium"
    else:
        return "Low"

In [ ]:
import numpy as np
import cv2

def draw_with_confidence(image, lines):
    output = image.copy()
    
    # 1. Get features and raw lines
    features, raw_lines = extract_features_from_lines(lines)
    
    if len(features) == 0:
        return output

    # 2. Get predictions from your RandomForest model
    # Note: Ensure your 'confidence_model' is already loaded!
    predictions = confidence_model.predict(features)

    left_scores = []
    right_scores = []

    # 3. 🔥 THE SPLITTING LOGIC
    for i in range(len(features)):
        slope = features[i][0] # Slope is the first feature
        score = predictions[i]
        
        if slope < 0:
            left_scores.append(score)
        else:
            right_scores.append(score)

    # 4. Calculate Averages
    # We use np.mean to get one stable number for each side
    left_avg = np.mean(left_scores) if left_scores else 0.0
    right_avg = np.mean(right_scores) if right_scores else 0.0

    # 5. Draw labels on the image
    # Left Side (Top Left)
    cv2.putText(output, f"L-Confidence: {left_avg:.2f}", (50, 50), 
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    
    # Right Side (Top Right)
    cv2.putText(output, f"R-Confidence: {right_avg:.2f}", (image.shape[1]-450, 50), 
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    return output

In [ ]:
def confidence_pipeline(image):
    
    color = HSL_color_selection(image)
    gray  = gray_scale(color)
    smooth= gaussian_smoothing(gray)
    edges = canny_detector(smooth)
    region= region_selection(edges)
    
    hough = hough_transform(region)
    
    result = draw_with_confidence(image, hough)
    
    return result

In [ ]:
import numpy as np

def extract_features_from_lines(lines):
    """
    Translates raw lines (coordinates) into mathematical features 
    for the Random Forest model.
    """
    features = []
    raw_lines = []
    
    if lines is not None:
        for line in lines:
            for x1, y1, x2, y2 in line:
                # 1. Slope: The angle of the line
                slope = (y2 - y1) / (x2 - x1) if (x2 - x1) != 0 else 0
                
                # 2. Intercept: Where the line hits the axis
                intercept = y1 - slope * x1
                
                # 3. Length: How long the line is (using geometry)
                length = np.sqrt((y2 - y1)**2 + (x2 - x1)**2)
                
                # We store these three numbers as 'features'
                features.append([slope, intercept, length])
                raw_lines.append([x1, y1, x2, y2])
                
    return np.array(features), raw_lines

In [ ]:
result = confidence_pipeline(image_to_use)

plt.imshow(result)
plt.title("Lane Confidence Visualization")
plt.axis("off")
plt.show()

## 9.Lane Confidance For Video

In [ ]:
from moviepy.editor import VideoFileClip
import os

# 1. Define your input and output paths
input_video = 'test_videos/challenge.mp4'
output_dir = 'output_videos'
if not os.path.exists(output_dir): os.makedirs(output_dir)
output_path = os.path.join(output_dir, 'smart_lane_output.mp4')

# 2. Load the video
clip = VideoFileClip(input_video)

# 3. Apply the "Smart" processor to every frame
# .fl_image is the magic command that loops through the video for you
processed_clip = clip.fl_image(smart_frame_processor)

# 4. Save the new video
%time processed_clip.write_videofile(output_path, audio=False)

In [ ]:
def smart_frame_processor(image):
    # 1. ORIGINAL VISION PIPELINE (OpenCV)
    # This finds the raw lines in the image
    color_select = HSL_color_selection(image)
    gray         = gray_scale(color_select)
    smooth       = gaussian_smoothing(gray)
    edges        = canny_detector(smooth)
    region       = region_selection(edges)
    hough        = hough_transform(region)
    
    # 2. MACHINE LEARNING (Random Forest)
    # This adds the L-Confidence and R-Confidence text to the image
    lane_result = draw_with_confidence(image, hough)
    
    # 3. DEEP LEARNING (YOLO)
    # This finds cars/trucks and draws boxes around them
    # verbose=False keeps your screen from being flooded with text
    yolo_results = model(lane_result, verbose=False)
    final_output = yolo_results[0].plot()
    
    return final_output

In [ ]:
from ultralytics import YOLO # Assuming you are using YOLOv8

# 1. Load your YOLO model (adjust the path to your best.pt if needed)
model = YOLO('yolov8n.pt') 

def smart_frame_processor(image):
    # --- TASK A: ROAD DETECTION (YOLO) ---
    # Run YOLO on the frame
    results_yolo = model(image, verbose=False)
    
    # Draw YOLO results (Boxes and Confidence) on the frame
    # .plot() is the easiest way to keep those confidence numbers!
    processed_image = results_yolo[0].plot() 

    # --- TASK B: DRIVER DETECTION (MediaPipe) ---
    # Convert for MediaPipe
    rgb_frame = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results_mp = face_mesh.process(rgb_frame)

    if results_mp.multi_face_landmarks:
        for landmarks in results_mp.multi_face_landmarks:
            ear_l = get_ear(landmarks.landmark, L_EYE)
            ear_r = get_ear(landmarks.landmark, R_EYE)
            avg_ear = (ear_l + ear_r) / 2.0

            # UI Overlay for Drowsiness
            color = (0, 255, 0) if avg_ear > 0.21 else (0, 0, 255)
            cv2.putText(processed_image, f"Driver EAR: {avg_ear:.2f}", (50, 50), 
                        cv2.FONT_HERSHEY_SIMPLEX, 1, color, 3)
            
            if avg_ear < 0.21:
                cv2.putText(processed_image, "DROWSINESS ALERT!", (50, 100), 
                            cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 255), 4)

    return processed_image

In [ ]:
# 1. Define where the video will be saved
output_video_path = 'output_videos/challenge.mp4'

# 2. Run the process
# We use output_video_path instead of the dots (...)
processed_clip.write_videofile(output_video_path, audio=False)

print(f"✅ Finished! Your smart video is saved at: {output_video_path}")

In [ ]:
from moviepy.editor import VideoFileClip

input_video = 'test_videos/challenge.mp4'
clip = VideoFileClip(input_video)

# Now Python knows exactly what 'smart_frame_processor' is!
processed_clip = clip.fl_image(smart_frame_processor)
processed_clip.write_videofile('output.mp4', audio=False)

In [ ]:
from IPython.display import Video

# This will play the processed video directly in your notebook
Video("output.mp4", width=700, embed=True)

## 10.Yolo Car Detection

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO
import os

# Load the model
try:
    model = YOLO('yolo11n.pt') 
    print("\n✅ Success! The AI 'Brain' is now loaded and ready.")
except Exception as e:
    print(f"\n❌ Still having trouble. Error: {e}")

In [ ]:
import matplotlib.pyplot as plt
import cv2

# 1. Load one of your project images (using OpenCV)
# Replace 'test_images/solidWhiteRight.jpg' with any image path you have
img_path = 'test_images/05. solidYellowLeft.jpg'
img = cv2.imread(img_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) # AI likes RGB colors

# 2. Let the AI "Predict" what is in the image
results = model(img)

# 3. Draw the boxes and labels on the image
annotated_img = results[0].plot()

# 4. Show the result using Matplotlib (which you already use in your project)
plt.figure(figsize=(12, 8))
plt.imshow(annotated_img)
plt.axis('off')
plt.title("YOLO Object Detection Test")
plt.show()

## 11.Drowsiness AlertFacial LandmarksEnhances driver safety/authentication.

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import winsound

# 1. Setup MediaPipe Face Mesh
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(refine_landmarks=True)

# 2. Eye Landmark Indices
L_EYE = [362, 385, 387, 263, 373, 380]
R_EYE = [33, 160, 158, 133, 153, 144]

def get_ear(landmarks, eye_indices):
    pts = [np.array([landmarks[i].x, landmarks[i].y]) for i in eye_indices]
    v1 = np.linalg.norm(pts[1] - pts[5])
    v2 = np.linalg.norm(pts[2] - pts[4])
    h = np.linalg.norm(pts[0] - pts[3])
    return (v1 + v2) / (2.0 * h)

cap = cv2.VideoCapture(0)
counter = 0

print("System Active. Press 'q' to quit.")

while cap.isOpened():
    success, frame = cap.read()
    if not success: break

    frame = cv2.flip(frame, 1)
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb_frame)

    if results.multi_face_landmarks:
        for landmarks in results.multi_face_landmarks:
            ear_l = get_ear(landmarks.landmark, L_EYE)
            ear_r = get_ear(landmarks.landmark, R_EYE)
            avg_ear = (ear_l + ear_r) / 2.0

            # 3. DROWSINESS LOGIC: EAR threshold (0.21) and frame count (20)
            if avg_ear < 0.21:
                counter += 1
                if counter >= 20:
                    cv2.putText(frame, "!!! DROWSINESS ALERT !!!", (10, 100), 
                                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 4)
                    winsound.Beep(1000, 200) # Warning sound
            else:
                counter = 0 

            cv2.putText(frame, f"EAR: {avg_ear:.2f}", (30, 30), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    cv2.imshow('VIT Project: Driver Safety Monitor', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()